# Kidder Isentropic Compression (1D)

This notebook runs the Kidder isentropic-compression benchmark: a shell of gas is driven inward (or outward) by boundary conditions consistent with an exact self-similar solution, so the run can be checked against an analytic density/pressure/velocity profile at every instant. `nu` (the geometry index) and `gamma = 1 + 2/nu` are tied to `dim`, not independent knobs -- the case is only isentropic for that pairing.

Two things make this case's step loop unlike `sod_1d.ipynb`'s or `02-linear-wave.ipynb`'s, and both stay visible as hook calls rather than being folded away:

- the inner/outer boundary bands are *driven* from the analytic solution after every step (`kidderCase.postStep`), not integrated;
- `dt` is re-derived from the state every step (`kidderCase.timestep`), because the shell compresses by more than an order of magnitude over the run -- which is also why the loop below is a `while t < tLimit` rather than a fixed `range(nSteps)`, exactly like `warpSPH.runner.run()`'s own time-limited branch.

`buildSystem` also only learns the true `tLimit` (`tauFraction * tau`, the analytic collapse time) once the system is sampled, so `ctx.spec` is re-read after that call rather than trusting the placeholder passed in.

Like `sod_1d.ipynb`, plotting calls `drawKidder` (the same per-frame redraw `kidderCase.setupPlot`/`updatePlot` use internally, exported directly from `warpSPH.cases.kidder` for this) rather than going through the `Case` hooks' `openWindow`/`pumpEvents`, which does not live-update reliably inside a Jupyter cell in this environment.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/03-Kidder_Isentropic_compression.gif)


In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float64', verbose=True)

from warpSPH import *
from warpSPH.cases.kidder import kidderCase, kidderStates, drawKidder
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `03-kidder-isentropic-compression.py`, made explicit and editable here.
# `kidderCase.defaults`/`kidderCase.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
# `nu`/`gamma` are not set here -- `kidderCase.configureScheme` derives them
# from `dim` below, because the case is only isentropic for that pairing.
spec = CaseSpec(caseName=kidderCase.name, scheme=kidderCase.scheme,
                params=dict(kidderCase.params)) \
    .merged(**kidderCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=100,
    dim=1,
    L=2.0,

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=100,
    store=False,

    # --- Kidder's own knobs (boundary bands, inner/outer states, ...) --------
    params=dict(
        band=10,
        r_inner=0.9, r_outer=1.0,
        P_inner=0.1, P_outer=1.0,
        rho_outer=0.01,
        tauFraction=0.99,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`kidderCase.buildSystem` -> `buildKidder`), not re-derived here.
# `configureScheme` derives `nu`/`gamma` from `dim` and mutates `spec.params`
# in place; `buildSystem` replaces `ctx.spec` outright once the analytic
# collapse time `tau` is known, so `spec` is re-read from `ctx` afterwards.
ctx = buildContext(kidderCase, spec)
kidderCase.configureScheme(ctx)
system = kidderCase.buildSystem(ctx)
spec = ctx.spec
solution = ctx.scratch['solution']
runningState = system.initializeNewState()

rhoInner, entropy = kidderStates(ctx)
print(f"r_inner = {ctx.param('r_inner')}, r_outer = {ctx.param('r_outer')}")
print(f"P_inner = {ctx.param('P_inner')}, P_outer = {ctx.param('P_outer')}")
print(f"rho_inner = {rhoInner}, rho_outer = {ctx.param('rho_outer')}")
print(f"s = {entropy}, tau = {solution.tau}, tLimit = {spec.tLimit}")


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct drawKidder + plt.subplots(), not kidderCase.setupPlot -- see the
# intro cell for why.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    fig, axis = plt.subplots(1, 3, figsize=(10, 5), squeeze=False)
    drawKidder(ctx, runningState, (fig, axis))
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = kidderCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=kidderCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
# `kidderCase.postStep`/`timestep` are what make this a `while t < tLimit`
# loop rather than a fixed `range(nSteps)` -- see the intro cell.
dt0 = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
storeSteps = max(1, int(spec.exportInterval / dt0)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
tq = tqdm(total=1000, leave=True)
i = 0
t = 0.0
while t < spec.tLimit:
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    kidderCase.postStep(ctx, runningState, i)
    ctx.config.dt = kidderCase.timestep(ctx, runningState)
    # -------------------------------------------------------------------------

    t = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = kidderCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=t))
    tq.n = min(1000, int(t / spec.tLimit * 1000))
    tq.set_description(f"t: {t:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))
    tq.refresh()

    final = t >= spec.tLimit
    if fig is not None and (i % spec.plotInterval == 0 or final):
        drawKidder(ctx, runningState, (fig, axis))
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or final):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=kidderCase.extraFields)

    i += 1
tq.close()


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
